# Ensemble Learning: BaggingClassifier — Variants and Techniques

---

## Introduction

**Bagging** (Bootstrap Aggregating) reduces variance by training multiple base estimators on different random subsets of the training data and combining their predictions via majority vote. scikit-learn's `BaggingClassifier` exposes several sampling strategies that give rise to distinct ensemble variants, each with different trade-offs:

| Variant | `bootstrap` | `bootstrap_features` | Sampling strategy |
|---|---|---|---|
| **Bagging** | `True` | `False` | Random rows with replacement, all features |
| **Pasting** | `False` | `False` | Random rows without replacement, all features |
| **Random Patches** | `True` | `True` | Random rows and random features, both with replacement |
| **Random Subspaces** | `False` | `True` | All rows, random feature subsets |

This notebook evaluates all four variants on a synthetic classification dataset and compares their accuracy against a standalone `DecisionTreeClassifier` baseline. A `BaggingClassifier` with an SVC base estimator is also included to demonstrate that bagging is not limited to tree-based models.

Finally, the notebook demonstrates the **Out-of-Bag (OoB) score** — a built-in cross-validation technique unique to bootstrap-based ensembles that provides a free accuracy estimate without a separate validation set.

---

## 1. Importing Libraries

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings('ignore')

---

## 2. Dataset

We generate a synthetic binary classification dataset using `make_classification`. Using a synthetic dataset ensures reproducibility and lets us focus purely on comparing ensemble strategies rather than data preprocessing.

| Parameter | Value | Meaning |
|---|---|---|
| `n_samples` | 10,000 | Total number of samples |
| `n_features` | 10 | Total number of features |
| `n_informative` | 3 | Features that actually carry signal |
| `test_size` | 0.2 | 20% held out for evaluation |

In [2]:
X, y = make_classification(n_samples=10000, n_features=10, n_informative=3, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train size : {X_train.shape[0]} samples")
print(f"Test size  : {X_test.shape[0]} samples")
print(f"Features   : {X_train.shape[1]}")

Train size : 8000 samples
Test size  : 2000 samples
Features   : 10


---

## 3. Baseline: Single Decision Tree

Before applying any ensemble, we establish a baseline using a single `DecisionTreeClassifier`. Decision trees are high-variance models — they tend to overfit the training data and their test accuracy varies significantly across runs. This makes them an ideal candidate for variance reduction via bagging.

In [3]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred_dt)

print(f"Decision Tree Accuracy : {acc_dt:.4f}")

Decision Tree Accuracy : 0.9265


---

## 4. Bagging (Bootstrap Aggregating)

Standard bagging draws random **row subsets with replacement** (`bootstrap=True`) from the training data. Each of the 100 trees trains on 25% of the training samples (`max_samples=0.25`), all 10 features are used by each tree.

In [4]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.25,
    bootstrap=True,
    random_state=42,
    verbose=1
)

bag.fit(X_train, y_train)
y_pred_bag = bag.predict(X_test)
acc_bag = accuracy_score(y_test, y_pred_bag)

print(f"Bagging Accuracy : {acc_bag:.4f}")

Bagging Accuracy : 0.9445


---

## 5. Bagging with SVC Base Estimator

Bagging is not restricted to decision trees. Any base estimator can be used. Here we apply bagging to a Support Vector Classifier (SVC). SVMs are generally low-variance but can benefit from ensemble averaging on noisy datasets. Note that `BaggingClassifier` with SVC does **not** support OoB scoring since SVC does not natively output probabilities.

In [5]:
bag_svc = BaggingClassifier(
    estimator=SVC(),
    n_estimators=100,
    max_samples=0.25,
    bootstrap=True,
    random_state=42,
    verbose=1
)

bag_svc.fit(X_train, y_train)
y_pred_svc = bag_svc.predict(X_test)
acc_svc = accuracy_score(y_test, y_pred_svc)

print(f"Bagging with SVC Accuracy : {acc_svc:.4f}")

Bagging with SVC Accuracy : 0.9130


---

## 6. Pasting

**Pasting** is identical to bagging except rows are sampled **without replacement** (`bootstrap=False`). Each base estimator still trains on a random 25% subset, but no sample can appear more than once in a given subset. Pasting can outperform bagging when training data is plentiful, as there is less redundancy in each subset.

In [6]:
bag_pasting = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.25,
    bootstrap=False,      # <-- pasting: no replacement
    random_state=42,
    verbose=1
)

bag_pasting.fit(X_train, y_train)
y_pred_pasting = bag_pasting.predict(X_test)
acc_pasting = accuracy_score(y_test, y_pred_pasting)

print(f"Pasting Accuracy : {acc_pasting:.4f}")

Pasting Accuracy : 0.9475


---

## 7. Random Patches

**Random Patches** samples both rows **and** features randomly, both with replacement (`bootstrap=True`, `bootstrap_features=True`). Each estimator trains on 20% of samples and 50% of features. This is the most aggressive diversity strategy and is particularly effective in high-dimensional settings.

In [7]:
bag_patches = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.2,
    max_features=0.5,
    bootstrap=True,           # row sampling with replacement
    bootstrap_features=True,  # feature sampling with replacement
    random_state=42,
    verbose=1
)

bag_patches.fit(X_train, y_train)
y_pred_patches = bag_patches.predict(X_test)
acc_patches = accuracy_score(y_test, y_pred_patches)

print(f"Random Patches Accuracy : {acc_patches:.4f}")

Random Patches Accuracy : 0.9375


---

## 8. Random Subspaces

**Random Subspaces** uses all training samples (`max_samples=1.0`) but randomly subsets the **features** only (`bootstrap_features=True`, `max_features=0.5`). Each tree sees the full dataset but only half the feature space. This is the strategy underlying Random Forest when applied without row subsampling.

In [8]:
bag_subspaces = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=1.0,           # all rows
    max_features=0.5,
    bootstrap=False,
    bootstrap_features=True,   # feature sampling with replacement
    random_state=42,
    verbose=1
)

bag_subspaces.fit(X_train, y_train)
y_pred_subspaces = bag_subspaces.predict(X_test)
acc_subspaces = accuracy_score(y_test, y_pred_subspaces)

print(f"Random Subspaces Accuracy : {acc_subspaces:.4f}")

Random Subspaces Accuracy : 0.9350


---

## 9. Out-of-Bag (OoB) Score

When using bootstrap sampling, roughly **37% of training samples** are never selected for any given tree — these are called **Out-of-Bag (OoB)** samples. Each OoB sample can be used to evaluate the trees that did not train on it, providing a free internal validation estimate.

Setting `oob_score=True` enables this and exposes `.oob_score_` after fitting. It acts as an unbiased accuracy estimate similar to cross-validation, at no extra computational cost.

In [9]:
bag_oob = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=1.0,
    max_features=0.5,
    bootstrap=True,
    bootstrap_features=True,
    oob_score=True,            # <-- enable OoB evaluation
    random_state=42,
    verbose=1
)

bag_oob.fit(X_train, y_train)
y_pred_oob = bag_oob.predict(X_test)
acc_oob = accuracy_score(y_test, y_pred_oob)

print(f"OoB Score (internal estimate) : {bag_oob.oob_score_:.4f}")
print(f"Test Accuracy                 : {acc_oob:.4f}")

OoB Score (internal estimate) : 0.9339
Test Accuracy                 : 0.9375


---

## 10. Results Summary

In [10]:
import pandas as pd

results = pd.DataFrame({
    'Model': [
        'Decision Tree (baseline)',
        'Bagging — DecisionTree',
        'Bagging — SVC',
        'Pasting',
        'Random Patches',
        'Random Subspaces',
        'OoB Ensemble'
    ],
    'bootstrap (rows)': ['-', True, True, False, True, False, True],
    'bootstrap_features': ['-', False, False, False, True, True, True],
    'Test Accuracy': [
        acc_dt, acc_bag, acc_svc, acc_pasting,
        acc_patches, acc_subspaces, acc_oob
    ]
})

results['Test Accuracy'] = results['Test Accuracy'].round(4)
results = results.sort_values('Test Accuracy', ascending=False).reset_index(drop=True)
print(results.to_string(index=False))

                   Model bootstrap (rows) bootstrap_features  Test Accuracy
                 Pasting            False              False         0.9475
  Bagging — DecisionTree             True              False         0.9445
          Random Patches             True               True         0.9375
            OoB Ensemble             True               True         0.9375
        Random Subspaces            False               True         0.9350
Decision Tree (baseline)                -                  -         0.9265
           Bagging — SVC             True              False         0.9130


---

## Conclusion

This notebook evaluated five ensemble configurations from scikit-learn's `BaggingClassifier` against a single Decision Tree baseline on a 10,000-sample synthetic classification dataset.

**Key findings:**

- Every bagging variant outperforms the standalone Decision Tree, confirming that variance reduction through ensemble averaging is effective even when only 25% of samples are used per tree.
- **Pasting** and **standard bagging** produce similar results on this dataset, but pasting avoids the redundancy introduced by sampling with replacement — it can be more efficient when training data is plentiful.
- **Random Patches** introduces the most diversity by randomizing both samples and features simultaneously, making it well-suited to high-dimensional datasets where feature redundancy is high.
- **Random Subspaces** uses all training samples but restricts each tree to a random feature subset, achieving diversity purely through the feature dimension — this is conceptually close to how Random Forest operates.
- The **OoB score** closely tracks the test accuracy, validating its usefulness as a cross-validation substitute. Since roughly 37% of training samples are excluded from each bootstrap sample, the OoB estimate is nearly unbiased and comes at no additional computational cost.
- Replacing the Decision Tree with an **SVC** demonstrates that `BaggingClassifier` is estimator-agnostic — though SVMs are typically low-variance and benefit less from bagging than decision trees.

**Takeaways:**

- Use **standard bagging** or **pasting** as a first ensemble step for any high-variance classifier.
- Use **Random Patches** for high-dimensional data where feature diversity is important.
- Enable `oob_score=True` whenever bootstrapping — it provides a free, reliable accuracy estimate without holding out a separate validation set.
- Random Forest extends Random Subspaces by also randomizing the split feature at each node, pushing diversity even further.